***
# This lecture

Models for **classification**:

-   Logistic regression
-   Random forest
-   Decision trees (in lecture notes)
-   Support vector machines (in lecture notes)

***
# Logistic regression

-   Simplest setup: **binary** classifier with $y_i \in \{0, 1\}$
-   Probability of $y_i = 1$ is given by **sigmoid** function (logistic CDF):  
    $$
    p(\mathbf{x}_i) \equiv 
    \text{Prob}\bigl(y_i = 1 ~|~\mathbf{x}_i\bigr) 
        = \frac{1}{1 + \exp\bigl(-\left(\mu + \mathbf{x}_i'\bm\beta\right)\bigr)}
    $$
-   Sigmoid function maps any real $z = \mu + \mathbf{x}_i'\bm\beta$ into $(0, 1)$:


In [ ]:
import numpy as np
from scipy.stats import logistic
import matplotlib.pyplot as plt

zvalues = np.linspace(-6, 6, 50)

plt.plot(zvalues, logistic.cdf(zvalues), lw=2.0)
# Add horizontal and vertical lines
for y in (0.0, 0.5, 1.0):
    plt.axhline(y, ls='--', lw=0.75, c='black')
plt.axvline(0.0, ls='--', lw=0.75, c='black')
plt.xlabel('$z$')
plt.ylabel(r'$\sigma(z)$')
plt.yticks([0.0, 0.5, 1.0])
_ = plt.title('Sigmoid function (logistic CDF)')

-   **Loss function:** derived from log-likelihood (MLE) + penalty
    $$
    L(\mu, \bm\beta) = 
    - \underbrace{\frac{1}{N} \mathcal{L}(\mu,\bm\beta)}_{\text{scaled log-likelihood}} 
    + \underbrace{\frac{r(\bm\beta)}{C}}_{\text{regularization}}
    $$

    -   Regularization term $r(\bm\beta)$: L1, L2, L1 & L2, None
    -   Regularization strength governed by $C$: large $C$ $\Rightarrow$ small penalty

***
## Example: Predicting binary class membership

-   Stylized example: $y_i$ is a function of two features $(x_{1i}, x_{2i})$:
    $$
    \begin{aligned}
    y_i &= 
    \begin{cases}
        1 & \text{if }~ f(x_{1i}, x_{2i}) + \epsilon_i \geq 0 \\
        0 & \text{else} 
    \end{cases} \\
    f(x_{1i}, x_{2i}) &= \sin(2\pi x_{1i}) \cos(\pi x_{2i}) \\
    \epsilon_i &\stackrel{\text{iid}}{\sim} \mathcal{N}\left(0, \sigma_{\epsilon}^2\right)
    \end{aligned}
    $$

### Creating a demo dataset

In [ ]:
import numpy as np


def f(x1, x2):
    """
    True function for classification examples
    """
    return np.sin(2 * np.pi * x1) * np.cos(np.pi * x2)

In [ ]:
def create_class_data(N=100, sigma=0.2, rng=None):
    """
    Create synthetic data for binary classification examples.

    Parameters
    ----------
    N : int
        Number of observations to generate.
    sigma : float
        Standard deviation of the error term.
    rng : numpy.random.Generator, optional
        Random number generator to use.

    Returns
    -------
    X : array-like
        Feature matrix.
    y : array-like
        Binary response variable.

    """

    if rng is None:
        rng = np.random.default_rng(seed=1234)

    # Draw features from uniform distribution
    x1 = rng.uniform(0, 1, size=N)
    x2 = rng.uniform(0, 1, size=N)

    z = f(x1, x2)

    # Add noise to latent variable if noise variance is positive
    if sigma > 0:
        # Draw errors from normal distribution
        epsilon = rng.normal(0, sigma, size=N)
        z += epsilon

    # Convert latent variable to binary response variable
    y = (z >= 0).astype(int)

    # Stack features into matrix
    X = np.column_stack((x1, x2))

    return X, y

In [ ]:
# Sample size
N = 100

# Standard deviation of noise
sigma_eps = 0.2

# Create demo dataset for classification
X, y = create_class_data(N=N, sigma=sigma_eps)

In [ ]:
from lecture3_classifiers import plot_classes

# Plot sample
plot_classes(X, y)

***
## Fitting a simple model (linear index)

### Step 1: Train-test split

-   Use **stratification** to preserve relative frequency of class labels in training and test sets

In [ ]:
from sklearn.model_selection import train_test_split

# TODO: Split data into training and test sets
# X_train, X_test, y_train, y_test =

# TODO: Verify that distribution of classes is similar in training and test sets

### Step 2: Estimate logistic regression model (no regularization)

-   Estimate simplest model with two features:
    $$
    z_i = \mu + \beta_1 x_{1i} + \beta_2 x_{2i}
    $$

-   Implemented in [`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
-   Relevant arguments:
    - `scikit-learn` version $\geq$ 1.8
        - `l1_ratio`: Weight on L1 penalty
    - `scikit-learn` version $<$ 1.8
        -   `penalty`: Type of regularization to use
    - Both versions:
        -   `C`: Inverse regularization strength
        -   `max_iter`: may need to increase this from default value
        -   `solver`: select solver (depends on type of regularization)
        -   `random_state`: needed for some solvers which use RNG

-   `LogisticRegression` implements four different types of regularization:


| Penalty type | $r(\bm\beta)$                                                                             | `penalty` (version $<$ 1.8)    | `l1_ratio` (version $\geq$ 1.8) | `C`                      |
|--------------|-------------------------------------------------------------------------------------------|--------------------------------|---------------------------------|--------------------------|
| L1           | $r(\bm\beta) = \lVert\bm\beta\rVert_1 = \sum_{k=1}^K \|\beta_k\|$                         | `'l1'`                         | 1                               | desired inverse penalty  | 
| L2           | $r(\bm\beta) = \frac{1}{2} \lVert\bm\beta\rVert_2^2 = \frac{1}{2} \sum_{k=1}^K \beta_k^2$ | `'l2'`                         | 0                               | desired inverse penalty  | 
| L1 and L2    | $r(\bm\beta) = \rho \lVert\bm\beta\rVert_1 + \frac{1-\rho}{2} \lVert\bm\beta\rVert_2^2$   | `'elasticnet'`                 | $(0, 1)$                        | desired inverse penalty  | 
| None         | —                                                                                         | `None`                         | —                               | `np.inf`                 | 

In [ ]:
# TODO: Create and estimate Logistic regression model
# lr =

### Step 3: Visually assess model predictions

-   Visually inspect decision boundary (possible for 2D case, not possible in general)

In [ ]:
from lecture3_classifiers import plot_decision_boundary

# Create x-values used to evaluate decisions
xvalues = np.linspace(0, 1, 1000)

ax = plot_classes(X_train, y_train, X_test, y_test)
plot_decision_boundary(ax, xvalues, lr)
ax.set_title('Classification with logistic regression')

### Step 4: Assess model accuracy

In [ ]:
from lecture3_classifiers import plot_generic_confusion_matrix

plot_generic_confusion_matrix()

-   **Accuracy:** implemented in [`accuracy_score()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)
    $$
    ACC = \frac{TP + TN}{FP + FN + TP + TN}
    $$
-   **Precision:** implemented in [`precision_score()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html)
    $$
    PRE = \frac{TP}{TP + FP}
    $$
-   **Recall:** implemented in [`recall_score()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html)
    $$
    REC = \frac{TP}{FN + TP}
    $$
-   **F1 score:** implemented in [`f1_score()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html)
    $$
    F1 = 2 \frac{PRE \cdot REC}{PRE + REC}
    $$


In [ ]:
# TODO: Predict y on test sample
# y_test_pred =

# TODO: Compute accuracy
# TODO: Compute precision
# TODO: Compute recall
# TODO: Compute F1 score

### Step 5: Plot the confusion matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

# Plot confusion matrix from predicted values
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_test_pred,
    colorbar=False,
    cmap='Blues',
    text_kw={'fontsize': 12, 'fontweight': 'bold'},
).ax_.set_title('Confusion matrix for linear index')

***
## Fitting a model with polynomials

-   Estimate model with polynomial interactions:
    $$
    z_i = \mu + \beta_1 x_{1i} + \beta_2 x_{2i} + \beta_3 x_{1i} x_{2i} + \beta_4 x_{1i}^2 + \beta_5 x_{2i}^2 + \dots
    $$

### Step 1: Estimate logistic regression model

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline

# Maximum polynomial degree
degree = 5

# Create logistic regression model without regularization penalty
# lr =

# Create pipeline with polynomial features and logistic regression
pipe_lr = make_pipeline(
    PolynomialFeatures(degree=degree, include_bias=False),
    StandardScaler(),
    # TODO: Add Logistic regression estimator
)

# TODO: Fit logistic regression with polynomial features

### Step 2: Visually inspect decision boundaries

In [ ]:
# Create x-values used to evaluate decisions
xvalues = np.linspace(0, 1, 1000)
ax = plot_classes(X_train, y_train, X_test, y_test)
plot_decision_boundary(ax, xvalues, pipe_lr)
ax.set_title('Classification with logistic regression (polynomials)')

### Step 3: Compute accuracy metrics

In [ ]:
# Predict y on test sample
y_test_pred = pipe_lr.predict(X_test)

acc_test = accuracy_score(y_test, y_test_pred)
pre_test = precision_score(y_test, y_test_pred)
rec_test = recall_score(y_test, y_test_pred)

print(f'Accuracy on test sample: {acc_test:.3f}')
print(f'Precision on test sample: {pre_test:.3f}')
print(f'Recall on test sample: {rec_test:.3f}')

### Step 4: Plot confusion matrix

In [ ]:
# Plot confusion matrix from predicted values
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_test_pred,
    colorbar=False,
    cmap='Blues',
    text_kw={'fontsize': 12, 'fontweight': 'bold'},
).ax_.set_title('Confusion matrix for polynomial features')

***
## Cross-validating the penalty term

- Regularization strength $C$ can be cross-validated with
  [`LogisticRegressionCV`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegressionCV.html)
- `LogisticRegressionCV` does not support pipelines

### Step 1: Perform cross-validation

In [ ]:
from sklearn.linear_model import LogisticRegressionCV

# Pipeline to create polynomial features and standardize them
transform = make_pipeline(
    PolynomialFeatures(degree=degree, include_bias=False), StandardScaler()
)

# Fit and transform training data
# X_train_poly = transform.fit_transform(X_train)


# TODO: Create and run Logistic regression cross-validation
# lrcv =

# TODO: Run cross-validation

# TODO: Store and report best C

### Step 2: Re-run model with optimal C (optional)

-   Not strictly needed, could directly use fitted `LogisticRegressionCV` object
-   But `LogisticRegressionCV` does not support pipelines...

In [ ]:
# Logistic regression model with optimal C from cross-validation
# lr_opt =

lr_pipe_opt = make_pipeline(
    PolynomialFeatures(degree=degree, include_bias=False),
    StandardScaler(),
    # TODO: Add Logistic regression estimator with optimal C
)

# TODO: Fit model
# lr_pipe_opt.fit(X_train, y_train)

### Step 3: Visually inspect decision boundaries

In [ ]:
# Create x-values used to evaluate decisions
xvalues = np.linspace(0, 1, 1000)
ax = plot_classes(X_train, y_train, X_test, y_test)
plot_decision_boundary(ax, xvalues, lr_pipe_opt)
ax.set_title('Classification with logistic regression (CV)')

### Step 4: Compute accuracy metrics

In [ ]:
# Predict y on test sample
y_test_pred = lr_pipe_opt.predict(X_test)

# Compute accuracy of cross-validated model on test data
acc_test = accuracy_score(y_test, y_test_pred)
pre_test = precision_score(y_test, y_test_pred)
rec_test = recall_score(y_test, y_test_pred)

print(f'Accuracy on test sample: {acc_test:.3f}')
print(f'Precision on test sample: {pre_test:.3f}')
print(f'Recall on test sample: {rec_test:.3f}')

### Step 5: Plot validation curve

- Plots a metric of model performance (e.g., accuracy) against the parameter $C$
- Use [`validation_curve()`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.validation_curve.html)
- `validation_curve()` supports pipelines

#### Create pipeline

In [ ]:
lr = LogisticRegression(
    fit_intercept=True, l1_ratio=0.0, max_iter=1000, random_state=1234
)
# NOTE: For sklearn version < 1.8, use penalty='l2' instead of l1_ratio=0.0
# lr = LogisticRegression(
#     fit_intercept=True, penalty='l2', max_iter=1000, random_state=1234
# )

pipe_lr = make_pipeline(
    PolynomialFeatures(degree=degree, include_bias=False),
    StandardScaler(),
    lr,
)

#### Compute validation curve

`validation_curve()` uses two arguments to specify which parameter should be varied:

-   `param_name` specifies the attribute which stores the parameter to be varied. For pipelines: `STEPNAME__ATTRIBUTE`
-   `param_range` specifies the range of parameter values to be varied

In [ ]:
from sklearn.model_selection import validation_curve

# Define range of values for C, spaced uniformly in logs
param_range = np.logspace(-4, 4, 20)

# Parameter name
param_name = 'logisticregression__C'

# Compute validation curve
# train_scores, test_scores =

#### Plot validation curve

- Use plotting function defined in `lecture3_classifiers` module

In [ ]:
from lecture3_classifiers import plot_accuracy_validation_curve

plot_accuracy_validation_curve(param_range, train_scores, test_scores)

***
# Random forest

-   Aggregates results from many decision trees ("model averaging")
-   Leads to less overfitting
-   Fully nonlinear classifier, usually does not require polynomial interactions, dummy variable encoding, etc.

### Step 1: Create estimation sample

-   Recreate estimation sample from earlier

In [ ]:
from sklearn.model_selection import train_test_split

# Sample size
N = 100

# Standard deviation of noise
sigma_eps = 0.2

# Create demo dataset for classification
X, y = create_class_data(N=N, sigma=sigma_eps)

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.3, random_state=1234
)

### Step 2: Estimate random forest

-   Implemented in
    [`RandomForestClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
-   Important hyperparameters:

    -   `n_estimators`: number of trees to grow
    -   `max_depth`: maximum depth of individual trees

-   Other important arguments:
    -   `random_state`: trees are grown on bootstrapped samples (involves RNG)
    -   `n_jobs`: number of parallel processes to use

In [ ]:
# TODO: fit random forest classifier
# forest =

### Step 3: Visually inspect decision boundaries

In [ ]:
ax = plot_classes(X_train, y_train, X_test, y_test)
plot_decision_boundary(ax, xvalues, forest)
ax.set_title('Classification with random forest')

### Step 4: Compute accuracy metrics

In [ ]:
from sklearn.metrics import accuracy_score

# Predict y on training and test samples
y_train_pred_forest = forest.predict(X_train)
y_test_pred_forest = forest.predict(X_test)

# Compute accuracy on training and test samples
acc_train = accuracy_score(y_train, y_train_pred_forest)
acc_test = accuracy_score(y_test, y_test_pred_forest)

print(f'Accuracy on training sample: {acc_train:.3f}')
print(f'Accuracy on test sample: {acc_test:.3f}')

## Cross-validating random forest hyperparameters

-   No dedicated cross-validation class for random forest is available
-   Use generic
    [`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)

### Step 1: Run grid search

-   Candidate grid for each parameter is specified using `param_grid` argument

In [ ]:
from sklearn.model_selection import GridSearchCV

# TODO: Define grid for max_depth

# TODO: Define grid for n_estimators

# TODO: Define dictionary of hyperparameters to search over

# TODO: Create and run GridSearchCV
# forest_cv =

In [ ]:
# Report best hyperparameters
print(f'Best accuracy: {forest_cv.best_score_:.3f}')
print(f'Best parameters: {forest_cv.best_params_}')

### Step 2: Visually inspect decision boundaries

In [ ]:
ax = plot_classes(X_train, y_train, X_test, y_test)
plot_decision_boundary(ax, xvalues, forest_cv)
ax.set_title(
    f'Classification with random forest (max depth: {forest_cv.best_params_["max_depth"]})'
)

### Step 3: Compute accuracy metrics

In [ ]:
y_train_pred_forest = forest_cv.predict(X_train)
y_test_pred_forest = forest_cv.predict(X_test)

acc_train = accuracy_score(y_train, y_train_pred_forest)
acc_test = accuracy_score(y_test, y_test_pred_forest)

print(f'Accuracy on training sample: {acc_train:.3f}')
print(f'Accuracy on test sample: {acc_test:.3f}')